# LPT diagnostic plots

This notebook generates the main diagnostic figures used for the LPT catalog.

## What is included
- Sky distributions in Galactic and equatorial coordinates
- RM-DM comparisons for the LPT sample
- RM-DM comparisons with ATNF pulsars overlaid
- A P-Pdot diagram with LPTs highlighted

> Sections that query the ATNF pulsar catalog require `psrqpy` and network access.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy import constants

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
})

lpts = pd.read_csv("LPTs.csv", na_values="-")
notes = lpts["Notes"].fillna("").astype(str).to_numpy()
is_binary = np.array(["binary" in note.lower() for note in notes], dtype=bool)

wd_idx = np.flatnonzero(is_binary)
unk_idx = np.flatnonzero(~is_binary)

N_LPT = len(lpts)
N_LPT_WD = len(wd_idx)
N_LPT_UNK = len(unk_idx)


def add_catalog_title(ax):
    ax.set_title(f"$N_{{\rm LPT}} = {N_LPT}$", pad=22, fontsize=14)
    ax.text(
        0.5,
        1.02,
        rf"{N_LPT_WD} WD-M dwarf binaries | {N_LPT_UNK} unknown progenitors",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=10,
    )


def binary_mask_from_notes(note_array):
    note_series = pd.Series(note_array).fillna("").astype(str)
    return note_series.str.contains("binary", case=False).to_numpy()


## 1. LPT distribution in Galactic coordinates

This figure uses the catalog Galactic coordinates directly and highlights the WD-M dwarf systems separately from the sources with unknown progenitors.


In [ ]:
l_deg = np.asarray(lpts["Gal_l (deg)"], dtype=float)
b_deg = np.asarray(lpts["Gal_b (deg)"], dtype=float)

l_wrapped = ((l_deg + 180) % 360) - 180
l_rad = np.radians(l_wrapped)
b_rad = np.radians(b_deg)

fig, ax = plt.subplots(
    figsize=(10, 5.5),
    dpi=300,
    subplot_kw={"projection": "mollweide"},
)

l_plane = np.linspace(-180, 180, 1000)
ax.plot(
    np.radians(l_plane),
    np.zeros_like(l_plane),
    color="black",
    lw=1.2,
    alpha=0.7,
)

ax.scatter(
    l_rad[unk_idx],
    b_rad[unk_idx],
    s=40,
    color="crimson",
    edgecolor="black",
    linewidth=0.6,
    zorder=5,
    label="Unknown",
)
ax.scatter(
    l_rad[wd_idx],
    b_rad[wd_idx],
    marker="s",
    s=40,
    color="teal",
    edgecolor="black",
    linewidth=0.6,
    zorder=5,
    label="WD-M dwarf",
)

ax.grid(True, alpha=0.4, linestyle="--")
xticks = np.arange(-150, 181, 30)
ax.set_xticks(np.radians(xticks))
ax.set_xticklabels([f"{x}°" for x in xticks])

ax.set_xlabel(r"Galactic Longitude $l$")
ax.set_ylabel(r"Galactic Latitude $b$")
add_catalog_title(ax)
ax.legend(loc="upper right", frameon=False, fontsize="small")

plt.tight_layout()
plt.savefig("images/lpt_distribution_galactic.png")
plt.show()


## 2. LPT distribution in equatorial coordinates

The catalog coordinates are converted from Galactic to FK5 before plotting on a Mollweide projection. Right Ascension is wrapped to keep the astronomical convention of increasing to the left.


In [ ]:
l = np.asarray(lpts["Gal_l (deg)"], dtype=float)
b = np.asarray(lpts["Gal_b (deg)"], dtype=float)

c_gal = SkyCoord(l=l * u.deg, b=b * u.deg, frame="galactic")
c_eq = c_gal.fk5

ra = -c_eq.ra.wrap_at(180 * u.deg).degree
dec = c_eq.dec.degree

ra_rad = np.radians(ra)
dec_rad = np.radians(dec)

fig, ax = plt.subplots(
    figsize=(10, 5.5),
    dpi=300,
    subplot_kw={"projection": "mollweide"},
)

ra_eq = np.linspace(-180, 180, 1000)
ax.plot(
    np.radians(-ra_eq),
    np.zeros_like(ra_eq),
    color="black",
    lw=1.2,
    alpha=0.7,
)

ax.scatter(
    ra_rad[unk_idx],
    dec_rad[unk_idx],
    s=40,
    color="crimson",
    edgecolor="black",
    linewidth=0.6,
    zorder=5,
    label="Unknown",
)
ax.scatter(
    ra_rad[wd_idx],
    dec_rad[wd_idx],
    marker="s",
    s=40,
    color="teal",
    edgecolor="black",
    linewidth=0.6,
    zorder=5,
    label="WD-M dwarf",
)

ax.grid(True, alpha=0.4, linestyle="--")
xticks = np.arange(-150, 181, 30)
ax.set_xticks(np.radians(xticks))
ax.set_xticklabels([f"{((-x) % 360) / 15:.0f}h" for x in xticks])

ax.set_xlabel(r"Right Ascension (J2000)")
ax.set_ylabel(r"Declination")
add_catalog_title(ax)
ax.legend(frameon=False, fontsize="small")

plt.tight_layout()
plt.savefig("images/lpt_distribution_equatorial.png")
plt.show()


## 3. RM vs DM for the LPT catalog

This view keeps only the LPT sample and displays the published uncertainties through error bars.


In [ ]:
DM = np.asarray(lpts["DM (pc/cm^3)"], dtype=float)
DM_err = np.asarray(lpts["DM_err"], dtype=float)
RM = np.asarray(lpts["RM (rad/m^2)"], dtype=float)
RM_err = np.asarray(lpts["RM_err"], dtype=float)

fig, ax = plt.subplots(figsize=(6.5, 5), dpi=300)

ax.errorbar(
    DM[unk_idx],
    RM[unk_idx],
    xerr=DM_err[unk_idx],
    yerr=RM_err[unk_idx],
    fmt="o",
    ms=6,
    color="crimson",
    ecolor="gray",
    elinewidth=1,
    capsize=2,
    markeredgecolor="black",
    markeredgewidth=0.6,
    zorder=3,
)
ax.errorbar(
    DM[wd_idx],
    RM[wd_idx],
    xerr=DM_err[wd_idx],
    yerr=RM_err[wd_idx],
    fmt="s",
    ms=6,
    color="teal",
    ecolor="gray",
    elinewidth=1,
    capsize=2,
    markeredgecolor="black",
    markeredgewidth=0.6,
    zorder=3,
)

ax.axhline(0, color="black", lw=1, ls="--", alpha=0.6, zorder=1)
ax.set_xlabel(r"Dispersion Measure (pc cm$^{-3}$)")
ax.set_ylabel(r"Rotation Measure (rad m$^{-2}$)")
ax.set_xlim([0, None])
ax.set_title("RM vs DM for LPTs", pad=10)
ax.grid(True, linestyle="--", alpha=0.4, zorder=0)

plt.tight_layout()
plt.show()


## 4. RM vs DM including pulsars

This section adds an ATNF pulsar background for context. Run the next cell only if `psrqpy` is installed and the ATNF query is available from your current environment.


In [ ]:
from psrqpy import QueryATNF

query = QueryATNF(
    params=["P0", "P1", "GL", "GB", "DM", "RM", "ASSOC", "BINARY", "TYPE", "BinComp"]
)
df = query.table

DM_psr = df["DM"]
RM_psr = df["RM"]

finite_rm_dm = np.isfinite(DM_psr) & np.isfinite(RM_psr)
DM_psr = DM_psr[finite_rm_dm]
RM_psr = RM_psr[finite_rm_dm]
GL = df["GL"][finite_rm_dm]
GB = df["GB"][finite_rm_dm]

print(f"ATNF rows with finite DM and RM: {len(DM_psr)}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5), dpi=300)

ax.scatter(DM_psr, RM_psr, marker=".", s=5, color="grey", zorder=1)

ax.errorbar(
    DM[unk_idx],
    RM[unk_idx],
    xerr=DM_err[unk_idx],
    yerr=RM_err[unk_idx],
    fmt="o",
    ms=6,
    color="crimson",
    ecolor="gray",
    capsize=2,
    markeredgecolor="black",
    markeredgewidth=0.6,
    zorder=3,
)
ax.errorbar(
    DM[wd_idx],
    RM[wd_idx],
    xerr=DM_err[wd_idx],
    yerr=RM_err[wd_idx],
    fmt="s",
    ms=6,
    color="teal",
    ecolor="gray",
    capsize=2,
    markeredgecolor="black",
    markeredgewidth=0.6,
    zorder=3,
)

ax.axhline(0, color="black", lw=1, ls="--", alpha=0.6)

for B in [-5, -2, -1, 1, 2, 5]:
    DM_line = np.linspace(0, 1000, 500)
    RM_line = (B / 1.232) * DM_line
    ax.plot(DM_line, RM_line, color="gray", lw=1, ls=":", alpha=0.7)

ax.annotate(r"$-5\,\mu\mathrm{G}$", (150, -1000), color="b", fontsize="x-small")
ax.annotate(r"$-2\,\mu\mathrm{G}$", (500, -1000), color="b", fontsize="x-small")
ax.annotate(r"$-1\,\mu\mathrm{G}$", (900, -850), color="b", fontsize="x-small")
ax.annotate(r"$5\,\mu\mathrm{G}$", (180, 1000), color="b", fontsize="x-small")
ax.annotate(r"$2\,\mu\mathrm{G}$", (550, 1000), color="b", fontsize="x-small")
ax.annotate(r"$1\,\mu\mathrm{G}$", (900, 820), color="b", fontsize="x-small")

ax.set_ylim([-1100, 1100])
ax.set_xlim([0, 1000])
ax.set_xlabel(r"Dispersion Measure (pc cm$^{-3}$)")
ax.set_ylabel(r"Rotation Measure (rad m$^{-2}$)")
ax.set_title("RM vs DM")

plt.tight_layout()
plt.show()


## 5. RM vs DM colored by the inferred line-of-sight magnetic field

This variant keeps the same RM-DM plane but colors both pulsars and LPTs by the estimated $\langle B_\parallel \rangle$.


In [ ]:
finite_lpt_rm_dm = np.isfinite(DM) & np.isfinite(RM)

DM_lpt = DM[finite_lpt_rm_dm]
DM_err_lpt = DM_err[finite_lpt_rm_dm]
RM_lpt = RM[finite_lpt_rm_dm]
RM_err_lpt = RM_err[finite_lpt_rm_dm]
notes_lpt = notes[finite_lpt_rm_dm]

binary_lpt = binary_mask_from_notes(notes_lpt)
wd_idx_lpt = np.flatnonzero(binary_lpt)
unk_idx_lpt = np.flatnonzero(~binary_lpt)

fig, ax = plt.subplots(figsize=(6.5, 5), dpi=300)

norm = mcolors.Normalize(vmin=-20, vmax=20)
B_para_psr = 1.232 * RM_psr / DM_psr
B_para_lpt = 1.232 * RM_lpt / DM_lpt

ax.scatter(
    DM_psr,
    RM_psr,
    c=B_para_psr,
    cmap="coolwarm",
    s=5,
    norm=norm,
    zorder=1,
)
sc_lpt = ax.scatter(
    DM_lpt,
    RM_lpt,
    c=B_para_lpt,
    cmap="coolwarm",
    s=36,
    edgecolor="none",
    norm=norm,
    zorder=1,
)

ax.errorbar(
    DM_lpt[unk_idx_lpt],
    RM_lpt[unk_idx_lpt],
    xerr=DM_err_lpt[unk_idx_lpt],
    yerr=RM_err_lpt[unk_idx_lpt],
    fmt="o",
    ms=6,
    color="none",
    ecolor="gray",
    capsize=2,
    markeredgecolor="black",
    markeredgewidth=0.6,
    zorder=3,
)
ax.errorbar(
    DM_lpt[wd_idx_lpt],
    RM_lpt[wd_idx_lpt],
    xerr=DM_err_lpt[wd_idx_lpt],
    yerr=RM_err_lpt[wd_idx_lpt],
    fmt="s",
    ms=6,
    color="none",
    ecolor="gray",
    capsize=2,
    markeredgecolor="black",
    markeredgewidth=0.6,
    zorder=3,
)

ax.axhline(0, color="black", lw=1, ls="--", alpha=0.6)
for B in [-5, -2, -1, 1, 2, 5]:
    DM_line = np.linspace(0, 1000, 500)
    RM_line = (B / 1.232) * DM_line
    ax.plot(DM_line, RM_line, color="gray", lw=1, ls=":", alpha=0.7)

plt.colorbar(sc_lpt, label=r"$\langle B_\parallel \rangle$ ($\mu$G)")
ax.set_ylim([-1100, 1100])
ax.set_xlim([0, 1000])
ax.set_xlabel(r"Dispersion Measure (pc cm$^{-3}$)")
ax.set_ylabel(r"Rotation Measure (rad m$^{-2}$)")
ax.set_title(r"RM vs DM for LPTs with $\langle B_\parallel \rangle$ lines")
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


## 6. P-Pdot diagram

This final section reuses the ATNF query result (`df`) and overlays the LPT measurements on a standard pulsar P-Pdot diagram, together with age lines, magnetic-field lines, and representative death lines.


In [ ]:
if "df" not in globals():
    raise NameError("Run the ATNF query cell above before building the P-Pdot diagram.")

periods = df["P0"]
pdots = df["P1"]

finite_ppdot = np.isfinite(periods) & np.isfinite(pdots)
periods = periods[finite_ppdot]
pdots = pdots[finite_ppdot]
types = df["TYPE"][finite_ppdot]

positive_pdot = pdots > 0.0
periods = periods[positive_pdot]
pdots = pdots[positive_pdot]
types = types[positive_pdot]

LPT_P = np.asarray(lpts["Period (min)"], dtype=float) * 60
LPT_Pdot = np.asarray(lpts["Pdot (s/s)"], dtype=float)
LPT_Pdot_err = np.asarray(lpts["Pdot_err"], dtype=float)
LPT_Pdot_ul = lpts["Pdot_ul"].fillna(False).astype(str).str.lower().eq("true").to_numpy()

finite_lpt_ppdot = np.isfinite(LPT_P) & np.isfinite(LPT_Pdot)
LPT_P = LPT_P[finite_lpt_ppdot]
LPT_Pdot = LPT_Pdot[finite_lpt_ppdot]
LPT_Pdot_err = LPT_Pdot_err[finite_lpt_ppdot]
LPT_Pdot_ul = LPT_Pdot_ul[finite_lpt_ppdot]
notes_ppdot = notes[finite_lpt_ppdot]

positive_lpt_pdot = LPT_Pdot > 0
LPT_P = LPT_P[positive_lpt_pdot]
LPT_Pdot = LPT_Pdot[positive_lpt_pdot]
LPT_Pdot_err = LPT_Pdot_err[positive_lpt_pdot]
LPT_Pdot_ul = LPT_Pdot_ul[positive_lpt_pdot]
notes_ppdot = notes_ppdot[positive_lpt_pdot]

binary_ppdot = binary_mask_from_notes(notes_ppdot)
wd_idx_ppdot = np.flatnonzero(binary_ppdot)
unk_idx_ppdot = np.flatnonzero(~binary_ppdot)


In [ ]:
plt.figure(dpi=300, figsize=(6.4 * 7.5 / 5, 4.8))

plt.scatter(periods, pdots, marker=".", color="k", s=5, label="Pulsars")

magnetar_idx = np.where(types == "AXP")
plt.scatter(
    periods[magnetar_idx],
    pdots[magnetar_idx],
    marker="s",
    color="r",
    facecolor="none",
    s=25,
    label="AXPs/SGRs",
)

age_list_years = np.array([1e3, 1e6, 1e9, 1e12])
sec = 365.25 * 24 * 3600
P_line = np.logspace(-3, 5, 200)
for age_yr in age_list_years:
    tau = age_yr * sec
    Pdot_line = P_line / (2 * tau)
    plt.plot(P_line, Pdot_line, "--", color="green", linewidth=1, alpha=0.7)

    exponent = int(np.log10(age_yr))
    label = rf"$10^{{{exponent}}}$ yrs"
    plt.text(0.0012, 0.002 / (2 * tau), label, fontsize=10, color="green", rotation=15, rotation_mode="anchor")

B_fields = np.logspace(9, 15, 7)
for B in B_fields:
    Pdot_B_line = (B / 3.2e19) ** 2 / P_line
    plt.plot(P_line, Pdot_B_line, "-.", color="b", linewidth=1, alpha=0.7)

    exponent = int(np.log10(B))
    label = rf"$10^{{{exponent}}}$ G"
    plt.text(20, (B / 3.2e19) ** 2 / 100, label, fontsize=10, color="b", rotation=-15, rotation_mode="anchor")

light_c_cgs = constants.c.cgs.value
R_NS6 = 1.2
R_NS = 1.2e6
Msolar = 2e33
M_NS = 1.4 * Msolar
K = 4 * (3 * light_c_cgs**3) * 2 / (5 * 8 * np.pi**2)

xx = np.logspace(-3, 6, 3000)
chen1_ppdot_pd = 1 / K * ((2.2e12 * R_NS6 ** (-19 / 8) * xx ** (15 / 8)) ** 2) * R_NS ** 4 / M_NS / xx
zhangIII_ppdot = 1 / K * ((9.2e25 * R_NS ** (-9 / 4) * xx ** (7 / 4)) ** 2) * R_NS ** 4 / M_NS / xx
beta = 10
chen4_ppdot_tw_multi = 1 / K * ((9.2e10 * beta ** (-1 / 4) * R_NS6 ** (-2) * xx ** (3 / 2)) ** 2) * R_NS ** 4 / M_NS / xx

plt.plot(xx, chen1_ppdot_pd, linestyle="dashed", color="black", alpha=0.6, lw=0.5, zorder=-1)
plt.plot(xx, zhangIII_ppdot, linestyle="dashed", color="black", alpha=0.6, lw=0.5, zorder=-1)
plt.plot(xx, chen4_ppdot_tw_multi, linestyle="solid", color="black", alpha=0.6, lw=0.5, zorder=-1)
plt.fill_between(xx, chen4_ppdot_tw_multi, np.maximum(chen1_ppdot_pd, zhangIII_ppdot), color="grey", alpha=0.2)

wd_upper = (binary_ppdot & LPT_Pdot_ul).astype(bool)
wd_measured = (binary_ppdot & ~LPT_Pdot_ul).astype(bool)
unk_upper = ((~binary_ppdot) & LPT_Pdot_ul).astype(bool)
unk_measured = ((~binary_ppdot) & ~LPT_Pdot_ul).astype(bool)

plt.errorbar(LPT_P[wd_upper], LPT_Pdot[wd_upper], yerr=LPT_Pdot[wd_upper] * 0.9, uplims=True, fmt="s", color="teal", label="WD-M dwarf")
plt.errorbar(LPT_P[wd_measured], LPT_Pdot[wd_measured], yerr=LPT_Pdot_err[wd_measured], color="teal", marker="s")
plt.errorbar(LPT_P[unk_upper], LPT_Pdot[unk_upper], yerr=LPT_Pdot[unk_upper] * 0.9, uplims=True, fmt="o", color="red", label="Unknown")
plt.errorbar(LPT_P[unk_measured], LPT_Pdot[unk_measured], yerr=LPT_Pdot_err[unk_measured], color="red", marker="o")

plt.xlabel(r"Periods (s)")
plt.ylabel("Period derivatives")
plt.xscale("log")
plt.yscale("log")
plt.xlim([1e-3, 1e5])
plt.ylim([1e-24, 1e-6])
plt.legend(loc=2)
plt.tight_layout()
plt.savefig("images/PPdot_with_LPTs.png")
plt.show()
